In [ ]:
#!/usr/bin/env python3
# ids_pro.py — Real live IDS with professional UI
# Run locally with root: sudo python3 ids_pro.py

!pip install numpy==1.26.4 # Downgrade numpy for scipy compatibility
!pip install river
!pip install scapy

In [ ]:
!pip install scapy fpdf2

In [ ]:
#!/usr/bin/env python3
# ids_pro.py — Real live IDS with professional UI
# Run in Google Colab with share=True

import scapy.all as scapy
from scapy.all import AsyncSniffer
from scapy.layers.inet import IP, TCP, UDP, ICMP
from river import anomaly
from collections import defaultdict, Counter, deque
from datetime import datetime
from zoneinfo import ZoneInfo
import threading, time, os, tempfile
import plotly.graph_objects as go
import gradio as gr
import html
import statistics
from fpdf import FPDF

# -----------------------
# CONFIG
# -----------------------
TZ = ZoneInfo("Asia/Kolkata")
LOG_LIMIT = 15
MODEL_TREES = 30
MODEL_HEIGHT = 10
MODEL_WINDOW = 200
DEQUE_MAX = 500

# -----------------------
# ONLINE MODEL (River)
# -----------------------
model = anomaly.HalfSpaceTrees(seed=42, n_trees=MODEL_TREES, height=MODEL_HEIGHT, window_size=MODEL_WINDOW)
THRESH = 0.65  # fallback threshold (used until stats are available)
WARMUP_PACKETS = MODEL_WINDOW  # model must see this many packets before classifying

# -----------------------
# STATE (bounded deques to prevent memory leaks)
# -----------------------
conn = defaultdict(lambda: {
    "start": datetime.now(TZ),
    "count": 0,
    "src_bytes": 0,
    "serror": 0,
    "rerror": 0
})

LOGS = deque(maxlen=LOG_LIMIT)         # dict-based log entries for UI cards
LOG_DATA = deque(maxlen=LOG_LIMIT)     # structured log data for filtering
ATTACKER_IPS = Counter()               # counts of suspicious IPs
TRAFFIC_X = deque(maxlen=DEQUE_MAX)    # timestamps
TRAFFIC_Y = deque(maxlen=DEQUE_MAX)    # cumulative packet index
ATTACK_COUNTS = deque(maxlen=DEQUE_MAX)  # cumulative attack count
PROTO_CNT = Counter()                  # protocol counts

sniffer_obj = None       # AsyncSniffer instance
sniffer_running = False
sniffer_error = ""       # stores permission/startup error messages
packet_counter = 0       # total packets seen
attack_counter = 0       # total attacks seen
last_score = 0.0         # last anomaly score for threat meter
SCORE_HISTORY = deque(maxlen=DEQUE_MAX)  # rolling score buffer for dynamic threshold

# -----------------------
# UTILITIES
# -----------------------
def now_ts():
    return datetime.now(TZ).strftime("%Y-%m-%d %H:%M:%S %Z")

def safe_html(text):
    return html.escape(str(text))

def proto_name(proto_num):
    """Map IP protocol number to readable name."""
    return {1: "ICMP", 6: "TCP", 17: "UDP"}.get(proto_num, f"OTHER({proto_num})")

# -----------------------
# FEATURE EXTRACTION
# -----------------------
def parse_flags(pkt):
    if not pkt.haslayer(TCP):
        return "NONE"
    f = pkt[TCP].flags
    mapping = {
        0x02: "SYN", 0x12: "SYN-ACK", 0x14: "RST-ACK",
        0x04: "RST", 0x01: "FIN", 0x29: "XMAS"
    }
    return mapping.get(f, "OTHER")

def extract_features(pkt):
    if IP not in pkt:
        return None
    src = pkt[IP].src
    dst = pkt[IP].dst
    proto = pkt[IP].proto
    key = (src, dst, proto)
    c = conn[key]
    c["count"] += 1
    size = len(pkt)
    c["src_bytes"] += size
    if pkt.haslayer(TCP):
        flags = str(pkt[TCP].flags)
        if "R" in flags: c["rerror"] += 1
        if "S" in flags and "A" not in flags: c["serror"] += 1
    duration = (datetime.now(TZ) - c["start"]).total_seconds()
    protocol_type = {1:0, 6:1, 17:2}.get(proto, 3)
    return {
        "duration": duration,
        "protocol_type": protocol_type,
        "src_bytes": c["src_bytes"],
        "count": c["count"],
        "serror_rate": c["serror"]/c["count"],
        "rerror_rate": c["rerror"]/c["count"]
    }

# -----------------------
# PACKET HANDLER
# -----------------------
def handle_packet(pkt):
    global THRESH, packet_counter, attack_counter, last_score

    feat = extract_features(pkt)
    if feat is None:
        return

    # online score and learn
    score = model.score_one(feat)
    model.learn_one(feat)
    last_score = score
    SCORE_HISTORY.append(score)

    ts = now_ts()
    packet_counter += 1
    TRAFFIC_X.append(ts)
    TRAFFIC_Y.append(packet_counter)

    # decide intrusion — skip during warm-up
    if packet_counter <= WARMUP_PACKETS:
        # Model is still learning baseline — never flag
        is_intrusion = False
    else:
        # Dynamic threshold: mean + 2*stddev of recent scores
        if len(SCORE_HISTORY) >= 30:
            mu = statistics.mean(SCORE_HISTORY)
            sigma = statistics.stdev(SCORE_HISTORY)
            dynamic_thresh = mu + 2.0 * sigma
            THRESH = max(dynamic_thresh, 0.5)  # floor at 0.5 to avoid over-sensitivity
        is_intrusion = score > THRESH

    # protocol counter
    if IP in pkt:
        p = pkt[IP].proto
        PROTO_CNT[p] += 1

    # extract full packet metadata
    src = pkt[IP].src
    dst = pkt[IP].dst
    proto_num = pkt[IP].proto
    pname = proto_name(proto_num)
    src_port = "—"
    dst_port = "—"
    flag = parse_flags(pkt)

    if pkt.haslayer(TCP):
        src_port = str(pkt[TCP].sport)
        dst_port = str(pkt[TCP].dport)
    elif pkt.haslayer(UDP):
        src_port = str(pkt[UDP].sport)
        dst_port = str(pkt[UDP].dport)

    # structured data for filtering & PDF export
    log_entry = {
        "src": src, "dst": dst,
        "proto": pname, "proto_num": proto_num,
        "src_port": src_port, "dst_port": dst_port,
        "flags": flag, "score": score,
        "ts": ts, "is_intrusion": is_intrusion
    }
    LOG_DATA.append(log_entry)

    if is_intrusion:
        ATTACKER_IPS[src] += 1
        attack_counter += 1
        ATTACK_COUNTS.append(attack_counter)

    # Store dict for UI card rendering
    LOGS.append({
        "is_intrusion": is_intrusion,
        "src": src,
        "dst": dst,
        "score": score,
        "flag": flag,
        "ts": ts,
        "proto_name": pname,
        "sport": src_port,
        "dport": dst_port,
    })

# -----------------------
# SNIFFER CONTROL (AsyncSniffer)
# -----------------------
def start_sniffer():
    global sniffer_obj, sniffer_running, sniffer_error

    if sniffer_running:
        return build_status_html()

    sniffer_error = ""

    try:
        sniffer_obj = AsyncSniffer(
            prn=handle_packet,
            store=False
        )
        sniffer_obj.start()

        # Allow thread to initialize
        time.sleep(0.5)

        if not sniffer_obj.running:
            sniffer_error = "Sniffer thread failed to start."
            return build_status_html()

        sniffer_running = True
        return build_status_html()

    except Exception as e:
        sniffer_error = str(e)
        return build_status_html()


def stop_sniffer():
    global sniffer_obj, sniffer_running

    if not sniffer_running:
        return build_status_html()

    try:
        if sniffer_obj and sniffer_obj.running:
            sniffer_obj.stop()
        sniffer_running = False
        sniffer_obj = None
        return build_status_html()

    except Exception as e:
        sniffer_running = False
        sniffer_obj = None
        sniffer_error = str(e)
        return build_status_html()

# -----------------------
# CSS INJECTION
# -----------------------
GLOBAL_CSS = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

  /* --- Base overrides --- */
  .gradio-container {
    background: #000 !important;
    font-family: 'Inter', 'Geist', system-ui, sans-serif !important;
  }
  /* Kill Gradio's default card backgrounds */
  .gr-block, .gr-box, .gr-panel, .contain, .gap {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
  }
  /* Markdown headings */
  .gr-markdown h1, .gr-markdown h2, .gr-markdown h3, .gr-markdown p, .gr-markdown strong {
    color: #a1a1aa !important;
    font-size: 11px !important;
    letter-spacing: 0.08em !important;
    text-transform: uppercase !important;
    font-weight: 500 !important;
  }
  /* Buttons */
  .gr-button {
    background: #111 !important;
    border: 1px solid #262626 !important;
    color: #fff !important;
    border-radius: 8px !important;
    font-size: 13px !important;
    font-weight: 500 !important;
    padding: 6px 16px !important;
    height: 36px !important;
    max-height: 36px !important;
    min-height: unset !important;
    transition: border-color 0.15s, background 0.15s !important;
    font-family: 'Inter', system-ui, sans-serif !important;
  }
  .gr-button:hover {
    background: #1a1a1a !important;
    border-color: #404040 !important;
  }
  .gr-button.primary {
    background: #fff !important;
    color: #000 !important;
    border-color: #fff !important;
  }
  .gr-button.primary:hover {
    background: #e5e5e5 !important;
  }
  .gr-button.stop {
    border-color: #ef4444 !important;
    color: #ef4444 !important;
  }
  .gr-button.stop:hover {
    background: #1a0000 !important;
  }
  /* Dropdowns and inputs */
  .gr-input, .gr-dropdown, select, input[type="text"] {
    background: #111 !important;
    border: 1px solid #262626 !important;
    border-radius: 8px !important;
    color: #fff !important;
    font-size: 13px !important;
    font-family: 'Inter', system-ui, sans-serif !important;
  }
  label.svelte-1b6s6s, .gr-form label {
    color: #71717a !important;
    font-size: 11px !important;
    letter-spacing: 0.06em !important;
    text-transform: uppercase !important;
  }
  /* Plotly chart backgrounds */
  .js-plotly-plot, .plot-container {
    background: transparent !important;
  }
  /* File download */
  .gr-file {
    background: #111 !important;
    border: 1px solid #262626 !important;
    border-radius: 8px !important;
    color: #a1a1aa !important;
  }
  /* Dividers */
  hr { border-color: #1f1f1f !important; }
</style>
"""

# -----------------------
# HTML BUILDERS
# -----------------------

def build_status_html():
    status = "Live" if sniffer_running else "Stopped"
    dot_color = "#22c55e" if sniffer_running else "#71717a"
    ts = now_ts()
    err = ""
    if sniffer_error:
        err = (
            f"<div style='margin-top:8px;padding:8px 12px;border-radius:6px;"
            f"background:#1c1007;border:1px solid #92400e;color:#fbbf24;font-size:12px'>"
            f"{safe_html(sniffer_error)}</div>"
        )
    return f"""
    <div style="display:flex;align-items:center;gap:10px;
                padding:12px 16px;border-radius:10px;
                background:#111;border:1px solid #1f1f1f;">
      <span style="width:8px;height:8px;border-radius:50%;
                   background:{dot_color};flex-shrink:0;
                   box-shadow:0 0 0 3px {dot_color}22;"></span>
      <span style="font-size:13px;font-weight:500;color:#fff;">{status}</span>
      <span style="font-size:12px;color:#52525b;margin-left:auto;
                   font-variant-numeric:tabular-nums;">{ts}</span>
    </div>{err}"""

def build_threat_meter():
    recent_attacks = len(ATTACK_COUNTS)
    recent_total = max(1, len(TRAFFIC_Y))
    anomaly_rate = recent_attacks / recent_total
    unique_att = len(ATTACKER_IPS)

    threat_score = (
        (anomaly_rate * 0.5)
        + (min(unique_att, 10) / 10 * 0.3)
        + (last_score * 0.2)
    )
    pct = min(100, int(threat_score * 100))

    if pct == 0:
        level, color, bg = "None", "#52525b", "#141414"
    elif pct < 15:
        level, color, bg = "Low", "#22c55e", "#0a1a0a"
    elif pct < 45:
        level, color, bg = "Medium", "#f59e0b", "#1a1400"
    else:
        level, color, bg = "High", "#ef4444", "#1a0000"

    bar_segments = ""
    for i in range(20):
        filled = (i / 20 * 100) < pct
        seg_color = color if filled else "#1f1f1f"
        bar_segments += f'<div style="flex:1;height:4px;background:{seg_color};border-radius:2px;"></div>'

    return f"""
    <div style="padding:14px 16px;border-radius:10px;
                background:{bg};border:1px solid #1f1f1f;margin-top:8px;">
      <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:10px;">
        <span style="font-size:11px;color:#52525b;letter-spacing:.08em;text-transform:uppercase;">Threat Level</span>
        <span style="font-size:13px;font-weight:600;color:{color};">{level}</span>
      </div>
      <div style="display:flex;gap:3px;">{bar_segments}</div>
      <div style="display:flex;justify-content:space-between;margin-top:8px;">
        <span style="font-size:11px;color:#52525b;">Attack rate {anomaly_rate:.1%}</span>
        <span style="font-size:11px;color:#52525b;">{unique_att} attacker IPs</span>
      </div>
    </div>"""

def _card(is_intrusion, src, dst, score, flag, ts, proto_name, sport, dport):
    if is_intrusion:
        accent = "#ef4444"
        label = "Intrusion"
        label_bg = "#1a0000"
        label_color = "#ef4444"
        score_color = "#f87171"
    else:
        accent = "#22c55e"
        label = "Normal"
        label_bg = "#0a1a0a"
        label_color = "#22c55e"
        score_color = "#4ade80"

    bar_w = min(100, int(score * 100))
    return f"""
    <div style="padding:12px 14px;border-radius:10px;background:#111;
                border:1px solid #1f1f1f;border-left:3px solid {accent};
                margin-bottom:8px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;">
        <span style="font-size:11px;font-weight:600;letter-spacing:.06em;
                     background:{label_bg};color:{label_color};
                     padding:2px 8px;border-radius:4px;">{label}</span>
        <span style="font-size:11px;color:#52525b;font-variant-numeric:tabular-nums;">{ts}</span>
      </div>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:4px 12px;">
        <div>
          <div style="font-size:10px;color:#52525b;margin-bottom:2px;">Source</div>
          <div style="font-size:13px;color:#e4e4e7;font-variant-numeric:tabular-nums;">{safe_html(src)}</div>
        </div>
        <div>
          <div style="font-size:10px;color:#52525b;margin-bottom:2px;">Destination</div>
          <div style="font-size:13px;color:#e4e4e7;font-variant-numeric:tabular-nums;">{safe_html(dst)}</div>
        </div>
        <div>
          <div style="font-size:10px;color:#52525b;margin-bottom:2px;">Protocol / Flags</div>
          <div style="font-size:13px;color:#a1a1aa;">{safe_html(proto_name)} &middot; {safe_html(flag)}</div>
        </div>
        <div>
          <div style="font-size:10px;color:#52525b;margin-bottom:2px;">Ports</div>
          <div style="font-size:13px;color:#a1a1aa;">{safe_html(sport)} &rarr; {safe_html(dport)}</div>
        </div>
      </div>
      <div style="margin-top:8px;padding-top:8px;border-top:1px solid #1f1f1f;
                  display:flex;align-items:center;gap:6px;">
        <span style="font-size:10px;color:#52525b;">Anomaly score</span>
        <div style="flex:1;height:3px;background:#1f1f1f;border-radius:2px;">
          <div style="width:{bar_w}%;height:3px;background:{score_color};border-radius:2px;"></div>
        </div>
        <span style="font-size:11px;color:{score_color};font-variant-numeric:tabular-nums;">{score:.4f}</span>
      </div>
    </div>"""

def build_logs_html(proto_filter="All", ip_filter=""):
    if not LOGS:
        return """<div style="padding:24px;text-align:center;color:#3f3f46;
                              font-size:13px;border:1px dashed #1f1f1f;border-radius:10px;">
                    Waiting for packets...
                  </div>"""

    filtered = list(reversed(LOGS))

    if proto_filter and proto_filter != "All":
        filtered = [l for l in filtered if l.get("proto_name") == proto_filter]

    if ip_filter and ip_filter.strip():
        needle = ip_filter.strip().lower()
        filtered = [l for l in filtered
                    if needle in l.get("src", "").lower()
                    or needle in l.get("dst", "").lower()]

    if not filtered:
        return """<div style="padding:24px;text-align:center;color:#3f3f46;font-size:13px;">
                    No packets match filter.
                  </div>"""

    return "".join(_card(**l) for l in filtered)

def build_top_ips_html():
    if not ATTACKER_IPS:
        return """<div style="padding:24px;text-align:center;color:#3f3f46;
                              font-size:13px;border:1px dashed #1f1f1f;border-radius:10px;">
                    No suspicious IPs yet
                  </div>"""
    rows = ""
    total = max(1, sum(ATTACKER_IPS.values()))
    for ip, cnt in ATTACKER_IPS.most_common(8):
        pct = cnt / total
        rows += f"""
        <div style="margin-bottom:10px;">
          <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
            <span style="font-size:13px;color:#e4e4e7;font-variant-numeric:tabular-nums;">{safe_html(ip)}</span>
            <span style="font-size:12px;color:#71717a;">{cnt} hits</span>
          </div>
          <div style="height:3px;background:#1f1f1f;border-radius:2px;">
            <div style="width:{int(pct*100)}%;height:3px;background:#ef4444;border-radius:2px;"></div>
          </div>
        </div>"""
    return f"""
    <div style="padding:14px 16px;border-radius:10px;background:#111;border:1px solid #1f1f1f;">
      <div style="font-size:11px;color:#52525b;letter-spacing:.08em;
                  text-transform:uppercase;margin-bottom:12px;">Top Attackers</div>
      {rows}
    </div>"""

def build_metrics_html():
    total_pkts = len(TRAFFIC_Y)
    total_attacks = len(ATTACK_COUNTS)
    active_ips = len(conn)
    rate = f"{(total_attacks/max(1,total_pkts)*100):.1f}%"
    cards = [
        ("Total Packets", f"{total_pkts:,}", "#fff"),
        ("Intrusions", f"{total_attacks:,}", "#ef4444"),
        ("Active Connections", f"{active_ips:,}", "#fff"),
        ("Attack Rate", rate, "#f59e0b"),
    ]
    items = ""
    for label, val, color in cards:
        items += f"""
        <div style="padding:14px 16px;border-radius:10px;background:#111;
                    border:1px solid #1f1f1f;flex:1;min-width:120px;">
          <div style="font-size:11px;color:#52525b;letter-spacing:.08em;
                      text-transform:uppercase;margin-bottom:6px;">{label}</div>
          <div style="font-size:22px;font-weight:600;color:{color};
                      font-variant-numeric:tabular-nums;letter-spacing:-0.02em;">{val}</div>
        </div>"""
    return f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:4px;">{items}</div>'

# -----------------------
# Plotly figures — dark Vercel palette
# -----------------------
PLOT_LAYOUT = dict(
    template="plotly_dark",
    paper_bgcolor="#000",
    plot_bgcolor="#000",
    font=dict(family="Inter, system-ui", color="#71717a", size=11),
    margin=dict(t=36, b=24, l=40, r=16),
    height=260,
    xaxis=dict(showgrid=True, gridcolor="#1f1f1f", zeroline=False, tickfont=dict(size=10)),
    yaxis=dict(showgrid=True, gridcolor="#1f1f1f", zeroline=False, tickfont=dict(size=10)),
    showlegend=False,
)

def build_traffic_fig():
    fig = go.Figure()
    if TRAFFIC_X:
        x = list(TRAFFIC_X)[-120:]
        y = list(TRAFFIC_Y)[-120:]
        fig.add_trace(go.Scatter(
            x=x, y=y, mode="lines",
            line=dict(color="#3b82f6", width=2),
            fill="tozeroy", fillcolor="rgba(59,130,246,0.08)"
        ))
    fig.update_layout(title=dict(text="Traffic", font=dict(size=12, color="#a1a1aa")), **PLOT_LAYOUT)
    return fig

def build_attack_fig():
    fig = go.Figure()
    if ATTACK_COUNTS:
        fig.add_trace(go.Scatter(
            x=list(range(len(ATTACK_COUNTS))),
            y=list(ATTACK_COUNTS),
            mode="lines",
            line=dict(color="#ef4444", width=1.5),
            fill="tozeroy", fillcolor="rgba(239,68,68,0.06)"
        ))
    fig.update_layout(title=dict(text="Intrusions", font=dict(size=12, color="#a1a1aa")), **PLOT_LAYOUT)
    return fig

def build_proto_fig():
    labels = ["ICMP", "TCP", "UDP", "Other"]
    vals = [
        PROTO_CNT.get(1, 0), PROTO_CNT.get(6, 0), PROTO_CNT.get(17, 0),
        max(0, sum(PROTO_CNT.values()) - sum(PROTO_CNT.get(p, 0) for p in [1, 6, 17]))
    ]
    fig = go.Figure(data=[go.Pie(
        labels=labels, values=vals, hole=0.55,
        marker=dict(colors=["#06b6d4", "#3b82f6", "#a855f7", "#52525b"],
                    line=dict(color="#000", width=2)),
        textfont=dict(size=11, color="#d4d4d8")
    )])
    fig.update_layout(
        title=dict(text="Protocol", font=dict(size=12, color="#a1a1aa")),
        paper_bgcolor="#000", plot_bgcolor="#000",
        font=dict(family="Inter, system-ui", color="#71717a", size=11),
        margin=dict(t=36, b=24, l=16, r=16), height=260,
        legend=dict(font=dict(size=10, color="#71717a"), bgcolor="#000",
                    bordercolor="#1f1f1f", borderwidth=1),
        showlegend=True,
    )
    return fig

# -----------------------
# PDF INCIDENT REPORT
# -----------------------
def generate_pdf_report():
    """Generate a PDF incident report and return the file path."""
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # Title
    pdf.set_font("Helvetica", "B", 22)
    pdf.cell(0, 14, "IDS Incident Report", new_x="LMARGIN", new_y="NEXT", align="C")
    pdf.set_font("Helvetica", "", 10)
    pdf.cell(0, 8, f"Generated: {now_ts()}", new_x="LMARGIN", new_y="NEXT", align="C")
    pdf.ln(8)

    # Threat Summary
    anomaly_rate = len(ATTACK_COUNTS) / max(1, len(TRAFFIC_Y))
    unique_attackers = len(ATTACKER_IPS)
    threat_score = (
        (anomaly_rate * 0.5)
        + (min(unique_attackers, 10) / 10 * 0.3)
        + (last_score * 0.2)
    )

    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "1. Threat Summary", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 11)
    pdf.cell(0, 7, f"Threat Score: {threat_score:.4f}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"Anomaly Rate: {anomaly_rate:.4f}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"Unique Attacker IPs: {unique_attackers}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"Last Anomaly Score: {last_score:.4f}", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    # Counters
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "2. Traffic Statistics", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 11)
    pdf.cell(0, 7, f"Total Packets Captured: {packet_counter}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"Total Attacks Detected: {attack_counter}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"TCP Packets: {PROTO_CNT.get(6, 0)}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"UDP Packets: {PROTO_CNT.get(17, 0)}", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 7, f"ICMP Packets: {PROTO_CNT.get(1, 0)}", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    # Top Attackers Table
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "3. Top Attacker IPs", new_x="LMARGIN", new_y="NEXT")
    if ATTACKER_IPS:
        pdf.set_font("Helvetica", "B", 10)
        pdf.set_fill_color(40, 40, 40)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(25, 8, "Rank", border=1, fill=True, align="C")
        pdf.cell(80, 8, "IP Address", border=1, fill=True, align="C")
        pdf.cell(50, 8, "Hit Count", border=1, fill=True, align="C", new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", "", 10)
        pdf.set_text_color(0, 0, 0)
        for rank, (ip, cnt) in enumerate(ATTACKER_IPS.most_common(15), 1):
            pdf.cell(25, 7, str(rank), border=1, align="C")
            pdf.cell(80, 7, ip, border=1, align="C")
            pdf.cell(50, 7, str(cnt), border=1, align="C", new_x="LMARGIN", new_y="NEXT")
    else:
        pdf.set_font("Helvetica", "", 11)
        pdf.cell(0, 7, "No attacker IPs recorded.", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)

    # Recent Logs
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "4. Recent Log Entries", new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 9)
    if LOG_DATA:
        for entry in reversed(LOG_DATA):
            tag = "INTRUSION" if entry["is_intrusion"] else "NORMAL"
            line = (
                f"[{tag}] {entry['ts']}  "
                f"{entry['src']}:{entry['src_port']} -> "
                f"{entry['dst']}:{entry['dst_port']}  "
                f"Proto={entry['proto']}  Flags={entry['flags']}  "
                f"Score={entry['score']:.4f}"
            )
            pdf.cell(0, 6, line, new_x="LMARGIN", new_y="NEXT")
    else:
        pdf.cell(0, 7, "No log entries recorded.", new_x="LMARGIN", new_y="NEXT")

    # Save
    out_path = os.path.join(tempfile.gettempdir(), f"ids_report_{datetime.now(TZ).strftime('%Y%m%d_%H%M%S')}.pdf")
    pdf.output(out_path)
    return out_path

# -----------------------
# GRADIO UI
# -----------------------
with gr.Blocks(title="IDS — Network Monitor", theme=gr.themes.Base(
    primary_hue=gr.themes.colors.zinc,
    neutral_hue=gr.themes.colors.zinc,
)) as demo:

    # Inject CSS
    gr.HTML(GLOBAL_CSS)

    # ── Header ──
    gr.HTML("""
    <div style="padding:28px 0 20px;border-bottom:1px solid #1a1a1a;margin-bottom:20px;">
      <div style="font-size:12px;color:#52525b;letter-spacing:.12em;
                  text-transform:uppercase;margin-bottom:6px;">Network Security</div>
      <h1 style="margin:0;font-size:24px;font-weight:600;color:#fff;letter-spacing:-0.02em;">
        Intrusion Detection System
      </h1>
    </div>""")

    # ── Metrics row ──
    metrics_html = gr.HTML()

    # ── Controls bar ──
    with gr.Row(equal_height=True):
        start_btn = gr.Button("Start", variant="primary", scale=0, min_width=80)
        stop_btn = gr.Button("Stop", variant="stop", scale=0, min_width=80)
        refresh_btn = gr.Button("Refresh", scale=0, min_width=80)
        proto_dd = gr.Dropdown(
            choices=["All", "TCP", "UDP", "ICMP"],
            value="All", label="Protocol", interactive=True, scale=1
        )
        ip_search = gr.Textbox(
            placeholder="Filter by IP...", label="IP Filter",
            interactive=True, scale=2
        )

    gr.HTML('<div style="height:1px;background:#1a1a1a;margin:12px 0 16px;"></div>')

    # ── Main layout ──
    with gr.Row():
        # Left: status + logs
        with gr.Column(scale=2):
            status_html = gr.HTML()
            threat_html = gr.HTML()
            gr.HTML('<div style="height:16px;"></div>')
            gr.HTML('<div style="font-size:11px;color:#52525b;letter-spacing:.08em;text-transform:uppercase;margin-bottom:10px;">Detection Log</div>')
            logs_html = gr.HTML()

        # Right: charts + top IPs
        with gr.Column(scale=3):
            with gr.Row():
                traffic_plot = gr.Plot(show_label=False)
                attack_plot = gr.Plot(show_label=False)
            with gr.Row():
                proto_plot = gr.Plot(show_label=False)
                top_ips_html = gr.HTML()
            gr.HTML('<div style="height:1px;background:#1a1a1a;margin:8px 0 12px;"></div>')
            with gr.Row():
                export_btn = gr.Button("Export PDF Report", scale=1)
                pdf_download = gr.File(label=None, interactive=False, scale=2)

    # ── Button wiring ──
    start_btn.click(fn=start_sniffer, outputs=[status_html])
    stop_btn.click(fn=stop_sniffer, outputs=[status_html])

    def manual_refresh(proto_filter, ip_filter):
        return (
            build_metrics_html(),
            build_logs_html(proto_filter, ip_filter),
            build_top_ips_html(),
            build_traffic_fig(),
            build_attack_fig(),
            build_proto_fig(),
            build_status_html(),
            build_threat_meter(),
        )

    ALL_OUTPUTS = [metrics_html, logs_html, top_ips_html,
                   traffic_plot, attack_plot, proto_plot,
                   status_html, threat_html]

    refresh_btn.click(
        fn=manual_refresh, inputs=[proto_dd, ip_search],
        outputs=ALL_OUTPUTS
    )
    proto_dd.change(fn=lambda p, i: build_logs_html(p, i),
                    inputs=[proto_dd, ip_search], outputs=[logs_html])
    ip_search.change(fn=lambda p, i: build_logs_html(p, i),
                     inputs=[proto_dd, ip_search], outputs=[logs_html])
    export_btn.click(fn=generate_pdf_report, outputs=[pdf_download])

    demo.load(
        fn=lambda: manual_refresh("All", ""),
        inputs=None, outputs=ALL_OUTPUTS
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_4013/1739557224.py:694: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://94a793ff9fa8334c82.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
import gradio as gr
print(gr.__version__)

6.19.0
